# [1] Introduction and Setup
Cross-Sell Recommendation systems predict what additional products a user might want to buy. 
They are responsible for up to 35% of Amazon's revenue. 
We will implement 4 approaches: Collaborative Filtering, Association Rules, Content-Based, and Hybrid.


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
from mlxtend.frequent_patterns import apriori, association_rules
import warnings
import joblib

warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)


# [2] Data Loading and Preprocessing
We load the raw data, parse timestamps, and handle any anomalies like returned items, which shouldn't be recommended.


In [ ]:
# Load data
df = pd.read_csv('../data/sample_transactions.csv')

# Parse dates (required for temporal split later)
df['transaction_date'] = pd.to_datetime(df['transaction_date'])

# Exclude returned items from training data since we don't want to recommend products they disliked
df_valid = df[df['is_returned'] == 0]

print(f"Original rows: {len(df)}, Valid rows: {len(df_valid)}")


# [3] Customer-Product Matrix
A user-item matrix represents interactions. Rows are users, columns are items. 
It is extremely sparse because a typical user only buys a tiny fraction of the catalog.


In [ ]:
# Build pivot table: rows=customer_id, cols=product_id, values=quantity
# We fill NaN with 0 because absence of purchase means 0 quantity
matrix = pd.pivot_table(df_valid, values='quantity', index='customer_id', columns='product_id', aggfunc='sum', fill_value=0)

# Compute sparsity
num_elements = matrix.size
num_non_zero = np.count_nonzero(matrix)
sparsity = 1.0 - (num_non_zero / num_elements)

print(f"Matrix shape: {matrix.shape}")
print(f"Sparsity: {sparsity:.4%}")


# [4] Collaborative Filtering — Cosine Similarity
"Users who bought similar things have similar tastes." 
We use Cosine Similarity which handles sparsity better than Pearson correlation for zero-heavy data.
Cold-start problem occurs for brand-new users.


In [ ]:
# Compute cosine similarity between all users.
# Memory implication: O(N^2) where N is number of users.
user_sim = cosine_similarity(matrix)
user_sim_df = pd.DataFrame(user_sim, index=matrix.index, columns=matrix.index)

# Demo: Find similar customers for C0001
target_user = 'C0001'
if target_user in user_sim_df.index:
    sim_scores = user_sim_df[target_user].drop(target_user).nlargest(5)
    print(f"Top 5 similar customers to {target_user}:\n{sim_scores}")


# [5] Association Rule Mining (Apriori)
Market Basket Analysis. We look for rules like {antecedents} -> {consequents}.
Lift > 1 means the items are bought together more often than expected by chance.


In [ ]:
# Group by transaction to form baskets
baskets = df_valid.groupby(['transaction_id', 'product_id'])['quantity'].sum().unstack().fillna(0)

# Convert to True/False for apriori
basket_sets = baskets.map(lambda x: True if x >= 1 else False)

# Run apriori algorithm (min_support trade-off: too high = no rules, too low = memory crash)
frequent_itemsets = apriori(basket_sets, min_support=0.01, use_colnames=True)

# Generate rules from itemsets
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.2)
print(f"Found {len(rules)} rules with Lift > 1.2")
display(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head())


# [6] Content-Based Filtering
Recommend items similar to what the user already likes, based on item attributes (category, price).
Pros: Solves item cold-start. Cons: Over-specialization (filter bubble).


In [ ]:
# Build product feature matrix
prod_info = df[['product_id', 'product_category', 'unit_price']].drop_duplicates('product_id').set_index('product_id')

# One-hot encode category
cat_dummies = pd.get_dummies(prod_info['product_category'])

# Normalize price (important so price doesn't dominate cosine similarity)
price_scaled = StandardScaler().fit_transform(prod_info[['unit_price']])

# Combine features
item_features = pd.concat([cat_dummies, pd.DataFrame(price_scaled, index=prod_info.index, columns=['price'])], axis=1)

# Item-item similarity
item_sim = cosine_similarity(item_features)
item_sim_df = pd.DataFrame(item_sim, index=item_features.index, columns=item_features.index)

print("Item Similarity Matrix computed.")


# [7] Hybrid Ensemble
Combine predictions from multiple models using a weighted average. 
We normalize scores first so a model outputting [0, 100] doesn't overshadow one outputting [0, 1].


In [ ]:
# (Conceptual pseudo-code of what is implemented in src/recommender.py)
# def hybrid_recommend(user, weights={'collab':0.4, 'assoc':0.4, 'content':0.2}):
#     recs_collab = get_collab_recs(user)
#     recs_assoc = get_assoc_recs(user)
#     recs_content = get_content_recs(user)
#     
#     norm_collab = normalize(recs_collab) ...
#     final_score = norm_collab * w1 + norm_assoc * w2 + norm_content * w3
#     return sorted_items

print("Hybrid strategy effectively implemented in the source code module.")


# [8] Model Evaluation Framework
Evaluate using Temporal Split (predict future from past) and ranking metrics like NDCG@K and Precision@K.


In [ ]:
# Temporal split
df_sorted = df.sort_values('transaction_date')
split_idx = int(len(df_sorted) * 0.8)
train_df = df_sorted.iloc[:split_idx]
test_df = df_sorted.iloc[split_idx:]

print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")

# (Evaluation logic is detailed in src/evaluator.py)


# [9] Business Impact Analysis
Translate ML metrics to business value.


In [ ]:
# Assuming a baseline CTR of 1% and Conversion of 10%
# If our system shows recommendations to 5000 users daily:
daily_users = 5000
ctr = 0.03 # Improved CTR from personalized recs
conversion = 0.15 # Higher conversion intent
avg_order_value = df['total_price'].mean()

expected_daily_sales_from_recs = daily_users * ctr * conversion * avg_order_value
print(f"Estimated daily incremental revenue from Recommender: ₹{expected_daily_sales_from_recs:,.2f}")


# [10] Model Saving and Deployment Notes
Models are saved using joblib for use in the Streamlit application.


In [ ]:
# Example deployment workflow:
# 1. recommender = CrossSellRecommender()
# 2. recommender.fit(df)
# 3. joblib.dump(recommender, 'models/recommender.pkl')
print("Notebook Complete. Check app.py for the deployment frontend.")
